# Prepare the shared 20 Newsgroups corpus

This is a one-time preparation notebook. It downloads the complete 20 Newsgroups corpus, removes the standard message headers, footers and quoted text, applies the shared spaCy text-cleaning and lemmatization rules, and saves the processed documents with their original category labels.

Experiments still choose their own category subsets, train/validation/test splits and fitted vocabularies. This notebook does not make those experiment-specific decisions. It does not import project modules, so it can be run on its own from the repository.

The first download and preprocessing pass may take a few minutes. The downloaded source dataset is cached under `data/raw/`; the final reusable artifact is `data/20news_preprocessed.npz`. The project `.gitignore` excludes `data/`.

## 1. Set paths and reproducibility settings

The seed controls the ordering used by scikit-learn when it returns the full corpus. It does not create a train/test split.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_20newsgroups

SEED = 42
REPOSITORY_ROOT = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
DATA_DIR = REPOSITORY_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_PATH = DATA_DIR / '20news_preprocessed.npz'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository folder: {REPOSITORY_ROOT.resolve()}')
print(f'Download cache:    {RAW_DATA_DIR.resolve()}')
print(f'Processed output:  {PROCESSED_DATA_PATH.resolve()}')

Repository folder: F:\OneDrive\Academics\bits\IR\Project\Local
Download cache:    F:\OneDrive\Academics\bits\IR\Project\Local\data\raw
Processed output:  F:\OneDrive\Academics\bits\IR\Project\Local\data\20news_preprocessed.npz


## 2. Download and inspect the dataset

`remove=('headers', 'footers', 'quotes')` asks scikit-learn to strip those parts of each Usenet post before returning its text. The dataset is cached locally, so rerunning this cell in the same working folder should reuse the download. Category IDs and names are retained as metadata; they are not used to preprocess or train a model here.

In [2]:
print('Fetching the complete 20 Newsgroups dataset...')
dataset = fetch_20newsgroups(
    subset='all',
    remove=('headers', 'footers', 'quotes'),
    shuffle=True,
    random_state=SEED,
    data_home=str(RAW_DATA_DIR),
)
documents = list(dataset.data)
labels = np.asarray(dataset.target, dtype=np.int64)
category_names = list(dataset.target_names)

print('Dataset loaded.')
print(f'Documents: {len(documents):,}')
print(f'Categories: {len(category_names)}')
print(f'Empty raw documents: {sum(not text.strip() for text in documents):,}')

category_summary = pd.DataFrame({
    'category_id': range(len(category_names)),
    'category': category_names,
    'documents': np.bincount(labels, minlength=len(category_names)),
})
display(category_summary)

print('Example raw document (up to 1,000 characters):')
print(documents[0][:1000])

Fetching the complete 20 Newsgroups dataset...
Dataset loaded.
Documents: 18,846
Categories: 20
Empty raw documents: 515


,category_id,category,documents
0,0,alt.atheism,799
1,1,comp.graphics,973
2,2,comp.os.ms-windows.misc,985
3,3,comp.sys.ibm.pc.hardware,982
4,4,comp.sys.mac.hardware,963
5,5,comp.windows.x,988
6,6,misc.forsale,975
7,7,rec.autos,990
8,8,rec.motorcycles,996
9,9,rec.sport.baseball,994


Example raw document (up to 1,000 characters):


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!




## 3. Apply the shared text preprocessing

For each document, spaCy tokenizes and lemmatizes the text. We discard spaces, punctuation, numbers, non-alphabetic tokens, stop words and lemmas shorter than two characters. Remaining lemmas are lowercased and joined with spaces. The rules match those used in the original notebooks.

No vocabulary is learned here. In each experiment, `CountVectorizer` is fitted only on that experiment's training documents, then used to transform its other splits and queries.

In [3]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
except OSError as exc:
    raise OSError(
        "spaCy model 'en_core_web_sm' is missing. Install it with: "
        "python -m spacy download en_core_web_sm"
    ) from exc

print('Loaded spaCy model en_core_web_sm.')
print('Preprocessing documents in batches of 256...')
processed_documents = []
for document_number, doc in enumerate(nlp.pipe(documents, batch_size=256), start=1):
    tokens = []
    for token in doc:
        if token.is_space or token.is_punct or token.like_num or not token.is_alpha:
            continue
        if token.is_stop:
            continue
        lemma = token.lemma_.lower().strip()
        if len(lemma) < 2:
            continue
        tokens.append(lemma)
    processed_documents.append(' '.join(tokens))
    if document_number % 5000 == 0:
        print(f'Processed {document_number:,} / {len(documents):,} documents')

print(f'Preprocessing complete: {len(processed_documents):,} documents.')
empty_processed = sum(not text for text in processed_documents)
token_counts = np.asarray([len(text.split()) for text in processed_documents])
print(f'Empty documents after preprocessing: {empty_processed:,}')
print(f'Mean tokens per document: {token_counts.mean():.1f}')
print(f'Median tokens per document: {np.median(token_counts):.0f}')

sample_index = next((i for i, text in enumerate(processed_documents) if text), 0)
print(f'Example processed document (document {sample_index}):')
print(processed_documents[sample_index][:1000])

Loaded spaCy model en_core_web_sm.
Preprocessing documents in batches of 256...
Processed 5,000 / 18,846 documents
Processed 10,000 / 18,846 documents
Processed 15,000 / 18,846 documents
Preprocessing complete: 18,846 documents.
Empty documents after preprocessing: 567
Mean tokens per document: 79.5
Median tokens per document: 35
Example processed document (document 0):
sure basher pens fan pretty confused lack kind post recent pen massacre devils actually bit puzzle bit relieved go end non pittsburghers relief bit praise pens man kill devil bad think jagr show well regular season stat lot fo fun watch playoff bowman let jagr lot fun couple game pens go beat pulp jersey disappointed islanders lose final regular season game pens rule


## 4. Save the shared corpus for experiment notebooks

The compressed NumPy archive stores the preprocessed document strings, original integer labels and category names. It uses plain string and integer arrays, so it can be loaded with `allow_pickle=False`. The raw download cache and this processed artifact both live under the ignored `data/` directory.

In [4]:
if len(processed_documents) != len(labels):
    raise ValueError('Processed document count does not match label count.')

np.savez_compressed(
    PROCESSED_DATA_PATH,
    documents=np.asarray(processed_documents, dtype=np.str_),
    labels=labels,
    category_names=np.asarray(category_names, dtype=np.str_),
)

size_mb = PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2)
print(f'Saved: {PROCESSED_DATA_PATH.resolve()}')
print(f'Archive size: {size_mb:.2f} MiB')
print(f'Documents: {len(processed_documents):,}; labels: {len(labels):,}; categories: {len(category_names)}')

Saved: F:\OneDrive\Academics\bits\IR\Project\Local\data\20news_preprocessed.npz
Archive size: 10.58 MiB
Documents: 18,846; labels: 18,846; categories: 20


## What the experiment notebooks should do next

Load the saved arrays with `np.load(PROCESSED_DATA_PATH, allow_pickle=False)`. Then, inside each experiment, make its own category selection and stratified split, fit its own count vocabulary on training text, and transform its validation/test/query text with that fitted vectorizer. This keeps the shared text cleanup consistent while preserving each experiment's distinct design.